# SST-2 指令学习：Kaggle GPU + Ollama + Qwen3.5

不训练模型，用一段指令判断影评情感：正面 `1`、负面 `-1`，统计准确率。
已内置与本机实验相同的 **SST-2 验证集 872 条**、评测脚本及数据来源记录。
这是 validation，不能称作隐藏标签的 GLUE test 集。

在 Kaggle 新建 Notebook，使用 **File → Import Notebook** 上传本文件。打开右侧 Settings，
Accelerator 选择账户当前可用的 GPU（优先 **GPU T4 ×2**），Internet 设为 **On**。
GPU 不可用时先解决额度或设置问题；下面的 GPU 检查会停止执行。
双卡选项不保证两张卡都会参与推理，后面用 `ollama ps` 查看实际运行情况。

建议按单元格顺序执行。默认会安装 Ollama、下载 `qwen3.5:4b`、运行例句和前 50 条；
**全量 872 条默认跳过**。先看 50 条的速度，再决定是否开启全量。
原生 Ollama 调用与本地 TSV 读取只用 Python 标准库，无需安装 `openai`、`datasets` 或其他 pip 包。


## 1. 配置与工作目录

默认 `basic`、关闭思考、temperature=0，与 API 实验设置一致。
模型可改为 `qwen3.5:2b` 或 `qwen3.5:0.8b`；修改后重新执行模型下载及之后的单元格。
模型权重放在 `/tmp/teacher_sst2_models`，实验输出放在 `/kaggle/working/teacher_task`。
不把权重放入最终导出包。


In [ ]:
import base64, csv, hashlib, io, json, os, platform, re, shutil
import signal, subprocess, sys, time, urllib.error, urllib.request, zipfile, zlib
from datetime import datetime, timezone
from pathlib import Path

if not sys.platform.startswith("linux") or not Path("/kaggle/working").is_dir():
    raise RuntimeError("本 Notebook 应在 Kaggle Linux 会话运行；不要在 Windows 执行安装单元格。")

OLLAMA_VERSION = "0.40.3"
MODEL = "qwen3.5:4b"
PROMPT = "basic"
THINK = "off"
TEMPERATURE = 0.0
MAX_TOKENS = 64
WORKERS = 1
REQUEST_TIMEOUT = 600
RUN_FULL = False  # 检查 50 条速度后，改为 True，再执行全量单元格。

ROOT = Path("/kaggle/working/teacher_task")
MODEL_DIR = Path("/tmp/teacher_sst2_models")
DATA_DIR = ROOT / "data"
META_DIR = ROOT / "metadata"
RESULT_ROOT = ROOT / "results"
for directory in (ROOT, DATA_DIR, META_DIR, RESULT_ROOT, MODEL_DIR):
    directory.mkdir(parents=True, exist_ok=True)
SCRIPT = ROOT / "sst2_instruction_eval.py"
DATA = DATA_DIR / "sst2_validation.tsv"
HOST = "http://127.0.0.1:11434"
RUN_ENV = dict(os.environ)
RUN_ENV.update(OLLAMA_HOST="127.0.0.1:11434", OLLAMA_MODELS=str(MODEL_DIR),
               OLLAMA_NUM_PARALLEL="1", OLLAMA_CONTEXT_LENGTH="4096", OLLAMA_KEEP_ALIVE="30m")
print("模型：", MODEL, "；模板：", PROMPT, "；思考：", THINK)
print("输出：", ROOT, "；全量开关：", RUN_FULL)


## 2. 恢复内置脚本和验证集

这个单元格只解压三个明确列出的文件，并核对 SHA-256、样本数与标签数量。
重跑时保留已有相同文件与断点；如果同名文件被改动，会停止，避免无意覆盖。
内置数据原标签是 0/1，评测脚本会转换成 -1/1。


In [ ]:
EMBEDDED_ASSETS = {
  "sst2_instruction_eval.py": {
    "sha256": "187d2f88d294e70ab427c694dc0dff6f54b85e3eac3849cba43eef15fc0d5906",
    "bytes": 41027,
    "zlib_base64": "eNrVfft3E9e18O/+K06H22UJJNkG0qZaVbrSQG/4mgcXaHv7GV1d2RphFVlSNRKGCq1lQgy2sbGT8MaER3iVxA9CwssY/y+3Hkn+qf/C3Y9zZs6MRsJJ22+tj9sba2bO2ee1X2fvffbZ9pO+qlXuG8oV+szCMVE6URkpFnb1GIZx8OCh6E7RmDm7sXrXXry/8eJWa/nTxvfn/v7q2sbz2dbSYnP1dOPhbftLeDNjzz3avHFr4/l4Y+l7rtJa+qt99wEXsCfvNi4t2muPEcLpicanN6FKc/Vma+m2ffZM8/ZS8/zZv41/0tNjL7+wX16wl1/ZzycbN+Ybl67+bXx6ILqxtt688NAB11q/3ro9o/fsb+PYicbC1MbzKfvup/b6ROv+qdbK6cbF76C7PUIMxARA2Hw001o+tXn9jDiWzucy6UquWPj7q8m3fw7jvHGbITRunW0uvv77qyno0L9/8Lu9onntU1ExrYqAevyxcWV589p86/IcfILq9sRDeCOiA1AJYEBr8I97v3l2RlilfK6SMBDEYHxnf9IQMFmt02vNhXOt11/YZ1/iXCxdhjmF357pEGJnTHCTor9vQDRm7zQm5x3I2LHoQB80O9kvEqL13c3NG3dE9B3qycwAvGosfiVfDfCAAOSumGh9v9I6NW3f/Wvj/D177qvG5aeNha/tldfYhelb4peVkVzh6DtQXC+18epa6+wje/qhfQ/6Oi1XAt48ewwdgYVrLd+Al40b6+4cjN9ojZ/eeH7enr8Psy+iUYIs7Ffj9sSTxvkH9uTTv7+67hmQbFzAjDavfwdNN1cX7TNX7fM37QfnGpfONha+gf5ii2euNp5cJKyBRW8+mLXnzze/XublzphmyTLNo0LsgV8H8Ze9dKVx6YV4d/8+6B/MgPi4ZBbe3Sd4dDCHmwvjgDSilCuJXMGqpPN5UYQi6RxMHYAs5vPp0TQNC7qw8FJ8TC8YmL10Grt0/mbzwk0GKPrSpVzf8Ei6QtSxvLF6Tw6bJgXRZu4RTAosL6y5/fIL2Qq1iK3YE083Vi+pXtoTr+ylF40FWCDA02vUdGy4VCLspKeoZZaPmeW/jZ/64ENxsFLN5Irw+9gH8NRcnALSev4caeizNSAaaP0PuUKmOGaJ/cUxs3xwxMznCXtx8v4NGEF8z969+w/u3fvbFExY6rd7/wjYZFhHo7FYzBAB/7YhUiP1rn1hT802H11GvAJiujgJAJmpCMuq7Ezh1Jarw0h5KROoMFY6AXiRMUeL7QA/czBj85Ov7Ll7G6/PwX+RXBbO2UszQNyt9Qv29S+31ERB7Oxva2JqFt8C7W8BRIdhTzzcPP1QSBaypZ4wEUAZp1PbRGPqC0APxg0krVdzWwJVKheP5TJmWWFnNDpazJh58ecxs7Ar9lZ895Bs5J8CrT/29pDzsHNIa2VL4MtmqViueKevMT0tyqZVzVcswRygufpF48uFxuOzwOpAntivbrVuP+zpaSzPNVc/tZe+BB5uLzx0KgFL2Fh9at99sfH8HBIfVgIp8IjfM0L/0p6ftRcvvxP7k1Us5J3WASQsWePWMyBopF4No5A857/m5Witn20+PMe83Z6fQW55ZRnEWfOTFwAe39x6htWJsjbPzraefQa8EjnXjXH4rbc/bB3TRs/twxhBmAGk5sJNGB6ze6DczTsobOEHtAb8E1g5CAZ4bD34qvElkMULwBF8HL8MPYIfleJRsyAaF1fgN4yguXhBazllVUdH0+UTNAPAW1wpc6px+RZQKUsirt149qzx7Ezz5l83r3zvAh5fha+oF/Rky8VRkUplq5Vq2UylRG6U1jVdKBQrJFKtnh71rnyklC5bpnqGCVA/R9LWSD43pB6xY+p30VK/yk5Nqzh81Kw4TyecIpWRspnO5ApHnBe5UadWtZyHNmJmuVws+955+iXflc0/V0FM8wiHgQpMwmFLDfG9YrVQMcvqe2G4Wi6bhUqMp8Ip9pt9Bw4eSr338Yf7P9h7aO+eiDhEfdxfLOb3HjeHq5ViOSLG0jnZUCldwZlQtffDY0/PNpEI/oeqDKs+StpOts49BnQH6QP/bU48aEx/yowTcLYTlJ739767Z+8B4Ooh49dmvjgmchasoNAoF2Y2XQFJag2Xc0MwuLSopK2jEehurmxmxFiuMsI1StUKl5X8wxJGj4vlRrZarowAU4H5qpjHKzHxh3KuYgI4mLESTK7JldMlqF4q59IVM38CCo+W8iboTPDRFHJdYka459D7+z76bergob37Zd+zxTIAK1ggxgAJIoK563C6bGareYCUHipS/0xBMHBg6UJGDMOKUC+silmKDp2I4l9Px6nzIMBzBVHMAoBi9cgIMJxKUZgFq4qNinzxSG44nSeA6WHABgSphkW9fffgb6mf7xbS+RN/MakfR3LHgKBwLgStP0wisKVcAasey5ljBC4Dgy+P4kusYgGWAVoXYI6L+TTM34mY1lfjgAn4V8DBwBTkTVGojg4hG88KM0dzHx0goAMRXjZ4HDKhqCiYR4Bij5n8Vb7EBjXgpaKVozJOJ3BkPfsPAH4fOgijq/VIXk5I1/zmG7FTbN7+HuR3c/WqeA8W99/3H0JFpTk1CawWmSYowPdPAedDRn1xBTgU8FbcVNxH9rZ5dpogGkNpKzdsxEVIIusOYRwuHC5s27ZN7HPxNH640LZs/9Lp/pGTrqbXmU/8GA2edNmEszzu1B8ufFQUiqTM46V8ukA8VxTL4k9VwO8sYCS9AIJGukFijeG0tUHHefwPSRMwiTVsxCwMm3XxB1izXqvDVDA1mDSjv+oI+ICkgrgRjrThx1uEH68fNc8vS6GvMAMwoPH4E3tiEreSi/fwfySEGSGGixU/OhjwR+MJnVEEyyE17vD0VisNbMwzC+pL15HsxpF4UPsM7qFQcZm713rweWvq29bSa3vpGg+gXMybqX9kFD1tKqjxx2IV5CyiE+KDifIzi8wWRD5x3VFavTRShJWzAJGDZ+JHTUe9p2fP3g8/Th3c+9GhvR+9t5d4HXBgS2Rz+VHRiyiYZ51gBPZVwDvTw5UqcEyAaUkkJSxLj8IALDGGkmB4pJyzKqNpK1opm+1UQTIlD+oAkk2OCLBUToPISDM4pw2rAGItDt9LRSCOaFTksiIDPHoIiBWfRnOsrgjFz36/DxbE5WiG2kXCatWQGZkp0BPgwRipVEpWvA+3eDFVKAYzbUSEQSozFlIfotk8qDpGJGAYY8XyUbNsQem3oSbSGcgqeBp4u7/O5Q1WxQN7AB2AaUjnR4pWJT4wsHvXbk/7rnbe3rbW8oCn5Z/1uy3TVvTNLb/d/3Z/37EBve2PigXzh7ZZ76bz7IyJjdXzrNYCx7DnLoG2isyzMXkJGGsnJSf18Ucf/DH10e8+hDUtm7hCpVzeDJWN/xrcEU3+aiD0q/jhWP+O8K/+DTAg9euP/3PvHn/Jw4eHisfNzOHaYWt7iGqF3Wrw7nAdq7770cE/7D2Q+vDdA79lpapsQKFsDojusLUDfrKCcjKfHjLzJ0vAknNE0uGT8oPDZhG2tUPx2vCvTgagvygbqoAGSwKHOQXSPcn7ozB2bu9/7v9g33v7DrVPRA8DOzyEPME3iB34Acb8q3jOOhk/mQg7E+COH98Nxn7yt/FPkjgX/wY4AND3QZt/+PjAHpp9/3wq6XdSybaTbKQ62Xyw3Pjy1Em2Yp1sPJ2EJxeeMwYE/A8O4gd1IWwo+B2Guk2QBnMfatlzz3DjdnG9eXOxeeu7xvQVfmNPPARR0PzmPjw2V2+iaJucb1xd2hy/RjY8YX8+IwZ6UnvePfj+XuQ/wPJjo2nY75QBPUI143B1586BnUh9USQzeOwf2OV93K09ZrP9Gc/jwAA+Dhh14HM9GTPLNskUqcohlONxbDKMhsJKFfTuQXiK4KtknHmBYeBGXdrmYrHYL/ukmY7FN8toGEVjYby5Osl0ak+esV+Nw/aksTCFZch2BWCkkePZM9jGw8TDngWAqG03Gilxj4mtksqW4D/AwOVbYOKGat5A/k/9d4iEPtBGgClLAoiVacght2pEDISdWmXW5VRl2AiCYjVsQnGntGGEYzAjuVIorGCrZ7df/m6RssDj/fY2mjGXZuyVx607XzcmHzUuLW68PING529vwxQ5e3b+N0Qbm4jTJTUONYytjMLpLgPzdFeWNQACw5XfGD9oc5wihhKIHznQKE4ym/eiCa+jCLH1IiJ0a0UY1pbfbzx/SZVF6/bD5t2XDl9vTJENenJF2rVnGjNT+OXC1dbSbTL1YjPmcRDwcsgS1eYeke3f3U9QwWyubFVSpOoLUNbY0tJeluw654HkG1eWte0H8X3ZjJQCtehAXbTWr6K1mczPVBDEeyUltX/QXBvnHzbnzwDEjbUF6Bt5Fb5GfwU1BgKrceU8Wl9AaVy80Lg92bh1trW0QpBAQqoWhT17sXkO7VLsMGktzyIkrbarmvdJVJeQCBTMI0jMjdXVxqdz3CM0ESxe3ni+2JxZUE6O6zgbc/Mw45vjU5u3XyCzmr8P6io0B23Z8183n0zDIrVev4al0E1FckHkstNfh+D83CU8OJCMES8DXdAMST7nkE2hWJF1435UZgwzzNFS5YShrUpC5EFFDLHIjoGczeRg/xZiKGGGPAqlqPRgdCCJ7XBVM2+ZBFc1P0obMbcTg6Mxs5AJheNJSRSGOFw5XD5cIO6//b8PG71GuK2nIVBGEFjsSLlYLYUGwqCUIAfm9gaACA1qn0eBOGlB/wbzBZfXbN8u2Yzz5r89fAf7sX3bO9S+AIIW+QLpv5IX4aQT3FA46W9EK85voav5QtKlEppR+DDYnwRWKZuLITMOU1mqRENx2bCjXYGsqgyPhAiQNjPbxOb9S4D4TAdAEPbcJGNv48ILkID23H1JhhIbyQRweq25+qC5ugiSgqvba18ABJYsTDEK8aj3ODCkcDW0wYF4Mu7RmSrpXF6Ozz82TzncTeeGczgXHp0pZpnpMowP4XhrwJYjg25EE7kztuKZFbQ/8cxQTZ4+1YiLJ1DHadmLnOoflHBbgi0O4iqxTzJkqS+AJelyxUJbQwgQLyx+kuC19X+It2mUXmpLjw7ljlSLVcurehJzgqlxVDv/AAMnM6WqeTS4zlMKY6UaxbIXRHunj6XzVZx4/Cpnsz/s1PdMdkorMxBug+SYWRIMNJZHp1gojDgVcowwSIusGuIvqaC2A0OGpuBtfQW2tApeVhMI2WU3IYPkpEHkaxZCRBzElQYkIbviUQ4DBZnDCIBpbo0TvIlsECrD18mF2upALlTjh5NLJzLpxKs7U47LszXZbvS8iQrckW4N+93yb8b6fxzb1eBp7AF4rrZFhN20IcJfvCdScxKlSUH4Rk87yuYKFFZh9HTbymP8wTL6khqzSx037qiD5ovpTKpcHLNC6fIRS1M+UfQP4qY3Gax8DtaMXOY4dl4ZrwxaSVBmjRAqluF6RMAmJon7Lu5I48a9xsv51vJ3IJ+0HQhMFLYdAyRJp7Kw2XTXA502sBzorAl5y7gznjHzaGQincg4zKSI9WIW7Mlyx73TH6tYx7CfscrxippuI6IRf3oMZXnSeUFGZTTShBBmRMBAi+gHSxjVSjb6dtTKHQF4BXMMSTYBagQa2LJe1oNuKeresHUstgem9AC9CGUjbucTzq82To0UVguY5XosZ1nVIcushLgFUNHMfKaQHgXeATg+mAxggdYJK2Yeh81N1qjhiOrCXp/YvL1qz0yAHipUK7RZpmZAx71rT1728WBUCHKwOcdZNQtoA0Wdk7sR0CpAIntN7Ah0VnY/zNvNsGfDFDB02FK5xF82yDhyOLMD5gHgEBDYJ4XoNxbPEeeJiH7ctAWLgPY5QCNzTU4iLiQyI9iD3KZgGZ4GniZ0Sl/8DtCZQ4UmL6k9FFAb7qhqUPgn5XqAxHJmNiHkNDgrGjh0pzzwWFJDYKA4PvVeTds/ZYhOY6wbws6x+deXAf0C8oilSyXU3EOw/KpaxFmCcNjLS1EzxjkhhElF8H/4BIsEoOo92nj78aWsI302zgvvGNvH1nj2rHnhIezzaVUQdXFVhKPxtpalRUiGuE08hQ0aPMIOs3HrFWyataH+xSwXUzjfCU+XQM+mFafif381SUFrHC8lgKf8/dUUOlUodEyixHWMYLpxz155zHFrUBummqK/Hrr8BqODEoqZxpGmXLyIC8uxLgTYtZmM4krQEo1hp6VcxXfOYOgdlKi3wZGUbOkL47I/q1gtE85CP4gDSskL4OL66hFzHsmmRnMYBOBdryIsV+FYrlwsgEiugNBJV/OA/u//JrX3oz37P9730SHkacraP5KNMhAy9bvrUimf8IIlNyPKAwDqBAWQOJMve1wtedgsVcQ+KrK3vYMOQhnNVy/tlc9cqBQxdk0PVlOf0OI2eak1PsEhd/g6inIJowZgZw9rby+sSMzoNooMLr/ea1CFT1Sjo/l83xFAPJKvVmUn/OUIR5po+hn2j28v/SHXO+iOZBVrrn3WXF3YvLy0eecK7O3s589bD051IqaN5+daa2uCI1Ltu49b392DwdfQKWVfX7PPTDA0GWY3fQuG7SwWxkd8vyI8iyc2Ly7Yp+c4pscIZFOgA9EkyqadSFDqQ1/GPIbyWmDMYeAse3eXbbQEPKk8SA/JsJeu4LXzlNwKjTGJQTV+kdQpLZimSDRmLF812KugnQSWJoqhoxzgynyqxy/3YDidGZ+LB3UKmSXrpmJ4k96Q2qB42inJFmleCY4WratNqsMAsgYBZezQm/cqcQXxjuiP+9cE/wzGuUBS12nxQ0Q20lWh3R0TegBqR512GNR9S/wGECVPlB5yqCLsaLCbE7PNtaXm41U0eS4vw3wdNU8IePjb+KmNtaubd26gGHz6BKO0KAAXmfbiFXvhIb5hC/u3t+1X4/8zfh/+x5OJMWnLF+U6fDq38QJfcnCtfWpBqrs9PQcPvXvodwdT7wPXc32gu/sH4sJ4d/8+gR3B1pZfoBHy29sYbHdxUroWd/fvhGJ6D9Goen5VlPLpCuDcqMdNKuyJCXv8lVN3F9Tl7rWW1oElNG6c3rw673zeDZ+BZdk3xrFlDiifn+W+oE30q/HGzXuAK+iljFbLeRazHLuogOzE/gFWYXDB3CegJWEv73xNsbqXdSBUCyYTfqWPRyn2zcIAWgM9lKmxdLlgZkD6mRX0kSRR/ABjDPOXFHql0RSkQtJiH8ALx6hORYpA2KFR64hrUXe3qKTRu5A8YgyqKCVS9cJDgfJlLJ3JIHivglQqI8fJGoONufnm3ZdJUYMideDc2XzVGkkcKldN7CRjKEccv5fPARtyENMNo/58pmtkMioxHGktQ5lBUNlzn/HUK1zjzRFs/lO5AuxXUyHLzMOWQ8WfRoRyOEdEupRLAepFBC2M9IfAU/p4ihcH/RejJdTyq+QvYbeypny2CTYVZUj9+/Gi2B2fTxBLx7nGqWB0MVk84W93G40FmE45Z4LsvrZ56iya46/ON74/Fdq98xdhh7Ltqw+ZLYTeOn6cXl9aBMHXuPwUie3lFzJC/9ljQFfUKF9dsz9dt88g8Tutkds+hRoNkrhHzdXjImPwQD9CaiXCMXT24w6Ot6uO/x/1gIGdP4/1w/9REEA8PuC3cLmtDpI2lRom7EJhJacjtoeVr/fh63HGvVClXLUqKVDQEr9JgzjzzSeDcAEwOoYkviQcvFEDSLg4JXEkIf9GghUA3z9tlRK7I2L7dndQvp65aEyPEnHpNwcnJjRM19HaC0ZHca7s4jmA6EABXhigSmRSqlH+C3LekJquTon0MTVUzJwgYiTOhDYW6eaL+3UAH3i/WcExCznvQflLeCYB+lHUAppoY6nNHhXQInCAUBwmtLl6pbV01wmYR4Xh/il74onaCaOPwT5zFQZgYbRPJqhzNUP5SSnEpXKihOqXAbiENchWA11mM4yCY9TrPW1g8IxHClcAnVupo2OoSxBIBpXSmikW6nUciH5gQ/QJPqQBOtF/jJkFeTpGWxoOVZU8kuHHaW26MTnZPY1UYtjRmAysxYismAS8fTsDbVPbNb4VAxXgYAUwzCLWyHq81xcCiAwrfMSEUrAjQ3ugRRVS+AUYA2JDmwWJagFD0ZWPAP9EOgfLoKlOWeP9Q4f2ixpWr8Py1/T6g/g2WQdhw7vgjfXbjVN4MKiGW0UzPBjf1d+frKMTDTdqXl8LNaVLKCDyvC6hRkuaNxxXwe0uTyMyVicSSucAxqhpWekjJmIHbAUwFhAxrgp4gHyT4qMLGBDFrdSTQYzJcOneAa9xAkNjBeq79qquLXGlnFYUqVN/WKdIKuVdDx7koEGfqAqxcYYHuO2KRTwFMT1tL53GyPRbz+wXT/HgHikCG2tfdENdq+T0TOLoFjD01+nMAY4TD5DekmuFuJtoxWlnNkpud8C/gFmIlYqlkD4T4a0OhAJDtLgYOlF5t7V8RobSEA/AcNLzq43zUyzEMW7om9tepaSNyZOo9PrtHLXTaK1fRXerapG1X71FEcTMULt49i33hGMyWAlnLsyxHa31+dbtGW38wyPFHJsUYRJi/ITeZacAKrQJWSwmKUNTihxe70YMXXjNB3/SVrEA/DQlCYbsWTiTuiZqz33tltR4tnyjMSroB5BNG1TFsNC42aGoXsRwBVkVR6I1gBMQQTKH121s0JUiDvVDKzH5xJA9bcbdQbSzByML+rQ1AoXkvPJziqsElGdO4/IT1WfqbMT/XXY+AI4rVTrDai8j4dXdrQcJRt/Ww3PcUnQ8ZgnsHk0GLNWhedAezb4sUoPQD16+YQviKok/er9BNInb0IQDzXGk9oHQ2SEMp9Mu1qAqDTWClXAFUlPCvTi3TcidM80V7AiQn3y/0nyyitbl1a+a82fwSNur8eaNcX5srS1vrD6Fkmx5g2ltrX+JQUPPp9kox+dRZeeAYebNMtmwvEelYvvLxeMn3ufvoVo9TJE2NJitbBNYv9KcWg5DJ6+Ur62hai6fSfHX0HbVKZ/23aZuq9+8UKg5e1ZWvvf3gM99oSj3SlTP4huF6mhKRuAi5TpIUv+xangJpkqhIggUqee1qxowJe3zI8Wfgy4Rsgkn8GhdLFMdLVkhhBmOkacQxAH5CY3w1vY/YoQ8NFaiZrzH3Cl6SOrM6RI6nsk+14eNYZypx0uprSr9AZ74Z3cbpi8POSnLgaosDQPt0VaoHEMDSygcy5iekYR/pOKGVlHgVRhmxn6GwhEz9JZPEcC5+9fqdgJVZhjZKJQjOR5BpYQwUTUnH+s9bT65Ltsx1ftByRyT3XdigUoZzStgk6PPEKYSPrW3BWM46osGInVNP44ZQw2+02aCV1EGbpnBq01ikQLlDKnSBzkrMU4cNyYJsbu/nxQ/OQlkBKcmHD+8oxZ2m8guahfxYVaxlHG2cWG5MXOqi1IXCN9V2bJGzUWzuugA0dXPJHdhGyew/BoPse719Xj3bYVKrlA13zR5uzsELbVvzaSkxnhetjnTZAh9INjliUlWGmFC5HnzUjWf9xQztsaZsoZ/pEE7O++Y3kmItyRCKMr/pdgVPEbkTTErDxppaKfYvl1V+IHzyTN1oEpBu95tLPepHhdqEMEjCKKi3x34oBsRwYBzFhkmEZuIklATFMDCC3y4+YCZBS6VISCsypb5DVEI7ZhlrbAilC1jgqNSyFWFnsP6XJMIgp6KhYcKCW5yCgSl59kvL4A619k9F+SuA2hOTCmnoGDgKp8BGVy6Isf/S1QIWtoQH3NX0jAiDvEPXp34j+ryFnvhdZx3QFiDfUn23WuwE91YnW1++pTdsRpz4Y0dygoOJpGSkaNqavU3bX2ojnoVDtwBOcUc05oq12VH5HYoA5sOuSUKUn7aNkVuTfkFU1rAPrGKHdziZsiFoVeuS88QbBGL+WMmGqwpmMx1hbgaC/z12GIxlsxjvZAHBXs6GWGVH1RaxtsK6h+DW1AHAsl2gVEEypIw3pr8BkXTrdubj2bQUwh7cnQETX5tLz2VbsnLT1tLr8UHH3yoMsr429eCMWiitKJyfQswFGYH6IxMtNXw56wxPEcMPGN2vTkoo6ZeY4SUY3JAU6x0ecJ20c2QgyJNF1uB53F7uyfQ2Vi7BeABsnG40Bvg1gdJNjyawWRV6FkUfigJHQRaPk4/2by43li4ubF+w168Iqsdx1j6F+fs+fNoF3813ri0wvzQVQPkpONaMw7ieTPpl/Hh4Bb3wpI9DWeR/J3TtIMKCmtnakcMRQjfnGdYX6g56B415fLuro2Kq0dVWh0iTf4gktDNDKEfvePXp1H3mYaC3JidKTzsa7inmwOqe4e6RSm8FROYmOb5uJNFp2vwbbpaKabcFkOS86HpgViS7K3LnnLKYpPLajvcYjbbPv27+3/xM/dgHKfX2Xhxs7l0mY8WCWoTYxQuXm09fdL868vNy9/ps/2z3XKpVZdotTlbhIx86d+5G5sYBrrnNCnoDdXOG+LxLT4VtXhv89FdSQM0am1uQ75BZmH7KYcpu9If63dHnJAj5j70x7RRck6njefnRL9AI+WVL9A3ARx06ovW2jdA9K69Uy8PnV5fa168Zy+dRnuMlBbVAo065BESEo3k727rFREFCkFSYkUOiOKLMXBRQq2naqyFp2oatDpveaI1+gPfemGb3Us8VvzSiTzK9hZqhXpv3RMljicQq0MYKPtf70b/bzr6l/7oL2KpaBIDZumsKnRBxUpQJgM63JYCCX7ELHMIw48dcTcdUhGpgukgPU2Sh9bijAFdwVFwlHBj1LueqgWR2Fxb2nh+Hg+AvpzjXFT2xGPKKnW2tXyGSRWDeE6t2xOzjcsP+ARl49mz1vp14OscMYSBMZdWOJpq4/miFsKOk5nJHUGzEMWLehfdXXjNSETlIjLLTSptDedyCWmOsIrlCvIwi2JG0LJWSsPUFMtWImREyKxnhNus2zLVU8waSe9862chDNX1maDCsRHzuOymMuJgPD3WRGWSvwwOqqC5iDdOLqIFsiXdGDdciqQ8elIsZHNHyHxjDY+YlFFhJxu4CaHYMiPZtmPhUcYdDbccEw49+THB0R3N46jPyhw5g1qNZEQzUQcxfD3ZQpD9OOJzAWpiwQ+pm+VSHldwSEvqqDTjdZ1q5dzzBIYjbilJqRQhClM6fLRUJBJNV0bidC7CMbQNEh0RNTDaoQYe939yA1LUMQlQ0KyKpe83GVfpI9qEUkMnKnje0Q1vkUceEcW0A5FHgbmahYwlI52ckIRsFnUs4FoeS2CukDGPR5xThe4hAj5CFehYpMOFHePeZUM7Es5JrPYNY+A2LdAMVzaHFcGyOZQabzOFBm0vqdL/OfjxR3uoNG3qIuJ3hRw+ae/CXcwJODso3LVDZVEx0M0m8HtkKMomwJnzOD2f0E8AMOAdYoBi/xuzN+0bcxwKCrotqAKc3wyT/b14yuF6nbfy2wSmKIUt6vNFaA+9HHOPQORvvDxjz04DdIwNmJhsPljdvHoXmrFXXm+eRmMD2uTW1+zpW8BNW0sr9tpFZL1Xvm0s3tl4fhePJHyLwZa44/n+mX4MVT8WFBvDBGcSNwETB+O8/sn2RdcNffazbzfWl+y7szwrMZK2gjOzNhYecfYA9NOofrWW72H+wFfXYFOO/SXPevDRCCYJpyNbsNhKvNaMR4B2EemOkAc+vB95D4bsOUynLjr50X8UOuDu8soyZxRFpzkJSp4Joy2oxOkMmcjw3CIfVKGdK9srEMaN283rS5ywETEk2ByLrGoQAErBkySP9nDPVqkbe5Mec06Ys+jL8FHDIUz85Jsl6N/sk4012NF9Df0TSKqYCpdzU1J49Ezz1cvG7B2ckxc3MeRv6a+gVgKuS8QlummtX6XUkufhE4YN3LjtR1byyxCekUPGSA8FHhAjnsgILfvrkQ1oW5AZRUaKYymYmWI5g8igOa5ck4UKQz1cwKdoNCp4HQVMbaKGk9wLv3qTdUrx6mbDFPyNBHxvUh1hUtAoWcLcPXtxHkN8qKTSDNzCjBWDmhlJOyRO5/OEEfsTCLCQv5jMuKExVL1hFYAGDcuwolovx173qoO3wEzfEfCJNePeXn+XGEm17ngGdvbl5oWrzsCorDsqr83Oyb/gjEVPxEBtKbtaklIydB0ZZ9bAlmVahA7DkzkXgsfoB8rpQJzhoOuUlntSPqO3m15goEmtl5Oe9qqp6gXsgr+V3qRsh8Nhe9sjmQkjKeVp7i/EnDwqOSj5ZiUd6FvFIEVaNKiiNEdqE/1e1VE8tSFfGLqi6ZSWR2Dd0jhEKKqOygVUQc2qSsnQHAWIkhrwmSc6MagfF4QtFLrhy15Q7WdA4KfL5502BrNcJoFH3+rkCaxRHxNRzBHk7XQCj4S47UDLW/UVM8hAiD8SoDpbHH/TxCJQjwJb274dlxu99VDbFwVjcNrO4RMYWFOsFjIhtd59UFTsDvPO1jlo7qutcCGu0CSi91T+6tAiyP2UKulvOlQAnUpWD7vdeMfBr04dIgZhubMkmYsna4QXB30AKuVqYRiUXWemWZJKMzpnNQECOVIZMboD4hRCZOgmAziQWEjmz6Xlw89tNBRum1+JtzTD8re/IZ+53sEP/XUSdZb+7j0Osts7NO//tBWAWlDZSNpZFWIvHhEjjxt3B5Y+diRlgXwF7cFBGNk79do/m4zDu/w4rLwOrKYOW8e0PVtE+JmlT4YPF/MUBqSO2DPHiTDFc6gDk0REogD+kNhDJ924q+2xZW9ai0j7fGKWDJI++IsxXT9Ml+wJOiZvjBlbPiuP+phTAKQWntjKCz46xe5KdG++eNy85R5dHJOn6Wl2QcPVQmHku+JYCGcx3NN2KI/m3jMRb5Dpb5Tovna9BhQpLTDHkJQfPqZKKOO8jqhzi44E7MK9XRKnpiQGJJU/T8faiAgm1yB/nga+A00i9I4kpjK3DcbfAmWmK3jFOfUOK8RKOnnSULdRydVDMiw/guuf8tlBPI47OkAxCPKoTetjpZLCJjgRvuG3pjEVooIxqLHHZLtKrHK8i5o12KuETm8yHtudJY3L0vSpeh8+FhzVS8/0RfWl3GHdXPd6ZdWpOtpWUVmWQVzUSXVHXxzhEqzFs85P15dcYzMn6h81pfqDitKbZKWRftblpS2yaIeSWkFvv3mMbZXckWKtzr0UnXq55U7+mD5urYtMD5hOGVZwglfFQ2CABhFaHneF28hJlnEvQiDUuO7toD17EbogUeDFE/uGRDdXWCHG7czCpv7B5wE9xbM6q2dwQ6BoRtseWVrAmnKtUIzWYJsoSHoP92rENNh48rDx6VxStJZeYx4G5/6NbBYngFLGbJ6d2VidResCh1WdmbCX8HIHPZgLTewytHy5dXrNnnhCEfSySudOF7r0GY9q/7BOo1vMnv0OOcSFp/bkSpcOcz99EWPefjq6Xofdp9MNP/Ei3qFBanJFsz+jzW3z6rxD8nzxjT19y777QHjO09ovnvIp/WzZtEYE22A8XZN67Fb65bAb4WKiTBEwg042upcDrTvk/mx8Awv4iI1mreWLOGEvr3uaVvo7otQP64lDl6C8+zvm5rh04g1kfpb3Dv6eIg6kqLcnL3MeA7zkhkyeRlhz/LmOMelH9vub1Jl11y3WwWMWJPze6OZ1xZi27tI5r73BeLTOHmQJNSyzvOqnGtnLr71SUSued9pmhlSUYL9tWCXsOsqh6wRHPUIPA2IUBp0k1W8OVwis7gtf0Bws0nOivaFTU4FuTg3VfCEY7bPoObPQKUQAkYSng93l7T7kri1wegQ6upDCLMtaXwfjAzt5rMVqJZXJlT1JseS7sF4gNnoU/gvaOF6dohyJ5OpJFY9qHpp88UhKZtlSoPvQP012cb7Kx5vggZkJMltVNcCD5Hyq4h0QR1U23OJYqphFY4hSkGGL5XUkKpsCXzsEJXNx8sSofFPwE1NQ+rxhqrlwDDTwUas9lxR6VGQT2APsPf8czCV1lRNlhWPX7pyfyVdfswc5lWWOq8CaqozuGKQ9v/aCJ6JSzBT9dijMqMH7CJpCla9AzhljiRLzwasKX3lNpUrduajnRqMgPIBeYR/1dKR4qdT4Kl4+wmmH+cKn+2ea1y/xvWXaXVDodsd8xSSaQKQ5niSZo/TiVTKto5MJfQmXHzBw3cyuhqpQbTRn4VUhXiz3D7ZrYZzJUU5/izscJGoQNfgbhxr2Gdkx7IcnK8hiPvFY1BwoSvtkbwzGxzz7FtQhNLxTi/K7X/kDMT73mahJnokKpSYDatpxGPiiM/CafiTVZ7ylmVNn+//xoDRHxuQ572SQS949NYIjwRnpeO6mgjctUhAw/kcjZ4rWJYlPZtyI9/B7mx+XY9rkyFT3YpgpJY3nd5i8E9ATnQeE286/utHWAaG7W8jCxOo7q0uYUGXtsf3FrD35NbvHUD9eWkHv0o17Mo9MJ1eZGjmQJ56kD5nhWIomN5XCsHZYZd5y6xbpiszfUK14g34p57EmAs1MRIwU2fHvpiWXpTkhKLXusVqEVOpodgG2Rx1zJiaaYWkRcVIq0UvXOELWjzh3JDDAWIuh0AV+UGSwMhcjMOLKvqbIZBLH4Ua02GVnipQZNgC23zaoISnwiEp/ROzcSrCz05T3w1bjnLXF9H8MAqFHdTtV3ZfkadYNXmrRB4OMN1ogufzlrH+cUcSXastLXXVmA5UUZYOVKWpTSlPz03zBRISE/6ZYzUeDDKxagZJ7QBXi0hxXQjeLUcKO0FsRl1uLvj4x0B9mn/SD5sv7aFGcXYF3gry51+2X9zHCd+IOCZin9vjV1tkn9spnjeU5sasfN9TaifBSsYhaavtNcyHkjJJDJ+RfKQ7MQobPQtewNgbnjeYqISwD+oxmjsbuSqmv8zHXsqq0HDTItltX2ZoKZbw8amwE06XJTsQDQjjs5/dbTydw5AsrzUU0S6lLS1HuzkyJ9yrl/I73RGMKJbI9N41FTy0Av2p3YRPdICNxB40X8IXko3sYcSDWH5FMIjU2YhYSvsv8woEp1bJV0nRUM53yO6PZDorGWCMK0dq7LFzArnRj9TxsSXGbOH3dPvsyEA7MpPTIa0F7FJ3RHrIXlvdLhTtCItkbCnfoMWlub4h/cNNoj2EQxECnj4pSdiQ8jm99OxdYNZeVpPVL1u/wId7RdMtFO/ZDFdFiFcKdWnXULYxrpRAOJHqg5FzBQ8lh9dHhBcQZOvcx4KgUXkVbg0psLKBLZV0rxsQkBzxxsCeedKfbLKCkyhMXHB/gG49CT+puCAcTdZjVOwnmUzK0R5EJatEeQaIxRfT4dzgLlsu6XFBNHi1M5zlh5TR4LSSsiJclwwAigbqYP35HXRch0TnGiVP9bhJnIzQ83NXZn6GcD320/JliUCgfM1dUh/xSmGULOebMscB6Jh2xRR29tX6ldf8UyAVRC7lCI4o1w2I7NdAnftYfjw1k68KePLP5+U3OMC2XTs/RHjzZ0oRVc0YSf2d3vU/bFSR1LwLMC/kOhOh2JM+1WfMMyhiPdj89NQjAajAQx0LcByhdgykI2hS0qby+GEV34wNCoLF4hzOEBgBS4hIYSbUC8qQQQlGgQpyHMZ4tL2+3bYsYtWIpOjY04OnTb80TQ8V0ObMPvdvlaqkS2Ktn3/ImEoMPYes5OY9WP55tRkzehuFZ3sUrIP/hm5SrdXn1sHbRsDRoahcNY9D3v2iwQOc97YBkoDaFn6AlQAkLR1R0tgz4rQKjjPY1o1ylIA0yPm0pOLvnTbHTkS2r6B1joj1WPAMoGNTUFPbToZRg7zVtG5xoIgrK6VCBlW1fbLk/UJw7iFEEZOD1gegUFN4tDLwn4KikmUmlsSd80rVSzhL3Mn76x+hPR6M/zYifvh//6Yfxnx40ZJSNNF4wy9QCrjjIStpgnUgDZROJaAFQHvMHF8X4edcKqqk5smzw6QQMGS1UErDPaVdBXQOI665VsFSnup+g+llMGpBatzHbKh6DnftMBVeeYqrEw8bdjlXxFeghac4KMqtjDmHaTEA5MxMiU6qyosaO5ItDIWO71/YV9pxypPrt7KcmYdQFXbZ2pYOLGYB4W2w3oOI/b7dADBk8sNhoxgjrS2hIuxrfcDa3zBOITAyYHPFAx9IGSmqnZfM5v1U25aQTK1di5dszchcvtUj5Eh8boL61tRVuMxLjZi6G2xxpK6eYbAAxWjxm8h0OIcO3GKhvS+N0uC1noiypb5ApaIb4LupaoTbLdZuo9di3McNeCO8pSI0imaYKFmhlpYDXvgVsSwjbevaktX7WXrmGkea0MHie/tm38LKxcBM3oLRyIK05Ll33rbFdCC88A2l0/bvGpRWf+tl2tAFXUGXrl5Pi2OFBv0cSwRye+fToUCYtrLgIWYOuNEAX16AUA/xbsk9+kN7XiIywkMvqXIZlnBQyUfNJITNTnJRnBPEHeUnxB19QclJgqqnJeT6ohe/JRIzXLJzUlCN4z1oPvCQvuNRl4Fk/02jhM4XANldvtpbgsz/ExTgZjUY7/X+8039OGi4tWErgaoHBpNEa/zN+wZA+Tifq0Rfrg+lG2oJFDC2ZURsgLSotGFZbJICh2QAxOBAhRlR0E1SCla8naqSj1znGkPK48gU4GODJnNGSZio9zDAianXH06LRs0pcUymraiRJIzyUsHbj2km6MvPwSf0WNuJMhtCT26nTZFlduCZqDLxXe9cbEb3QRi9eOeMKZaek+8otaHhPMyk6yQLiyjgOIgP0K/MbogT3kYnBfSZ6kI84bPrBI1BFCvy5g16POd2Hh1VZLfgEX5jOh+AAEm4MVwd+GmH9Yk+cWF51eYyITRU9rn28i5hRpXRpg//pKvqpBPovpTpOZ+bpchSAVHf87Blg8EF5GLZ+4PT/U1/5P9vvwuPo4HVp83Z4bhLXV81IGLDh/Xm/14vlHlQGzJBHleGXZOWeM8v4mhm7PLbsQQojiv8EaHPAqO3zqJ40Zs6CYBP0wVOWocrVCPIFeSykHi+PXvVNe1ntPgXPZlYqzMxPva2Tkb/Hn+USetAWB6kb1Om7FkPan3TPpshKbQFKPF366X3PRLklHRDqZGTArLMIRo2QAmQCprxbdweSqmSwk8hX3oM/3D6LYwqCQyAUGQlwVBwkHUCR97TSXTmc2aRx+SnxPlwGNsCAwG+PjJTRd9gLf/wd4qSKvqPvAeyzLrUGEkDtB1xG08A6fZzGg36IRRbs1cnizBwfCZ6NlAk3exr5xjlmzIlSWpiCbU7j9ERzcao59aKxNG0vXm5+c186wM8+2Xh5vjF9b/PC1Z4gD6N+T5pl8bQzx+LMlu+Wj1Qx6mU/PpVDGdMaBgTBmolUKlMcTqUigtlDxSynKDlowql8ID22x63wvpkv/UYVlSuMtyCk0rINxDRHd4zIzKhWgiyCTgwPHmzmbIwJN+k4YBQATxisL6IHlDLeCqdAp+bk1ROigLwzYexwIDm3V3BsHN5Wc+aqfffaxvNHG69voNvn2WeAYb52otk8noPvE38eMwu7Ym/Fdw8ZXUaKqrCnbf+QkRVrAx6UqTOC46Flx4khMm9t77vbZYbUqXOsmgf2bZDCOyMUMBlxU2Mm9W5iia6d1FJnYFa5bFZgTuM+TOSOqcX6FCy6HfnVLU6fB5t3pGgtpbE7HGyx02Cwn+jrTnBwl+xlv4M0c49wVzQ1KwoyeKJfXXdN1n1gNLDxbS2f2rx+Rrz9851caKpjc8QGDQ1NtUtxHPSiTYs9+bk9ecZF17bbc9qBO9cXuaA4wRPfjkinc6HDFetY3zBeHDXT9Ua+bsNw7mGCltKUgC5hWJViGVS4ctVtHhMZBV3ZxNcxcbdg4jo2o+6C0cZDsbt0jQzdNCxzVuHSX3jREU66lItisiYFRqaFwpuzV1+2lvAmY4DQPL9s3zltz13ZPDsnOmShCuASTkCsB5NUf8eBtnjzCT/wwhGZFkfiJqfhBnLESyy1bJB4Bcj0w83xzkug2wNlw5zWRDatiOCakzRZ9OOSq+sWMLNNR+AqfrJ9RGjAxoP4oGXdAp7BkbrtzE68TTem0yVlA53HIBM/BfXfnr3owJfTuDgFjWoR6O3UBTI/kJ53OVApWox99EDVAA8PkGPaZIwgd6ltV8c2oL9R2MjoNCyN4C7ak0FOpjm4+8KFqgp2gk0xb91JSiI7RXghBayet+9+5wm9DuALsBPqDhUTK8x91rgx37iEbmuYkM1PvrLn7m28Pgf/pYPc52SgHU1U56bKJ6Jo/X9Ta+wJlZwOUJ8EE98drt0zdAq1lOnHm58/7tgg7yW7t9eYnvYtCY+xMTWOBk2yXaJlbHIeGIr96hYweNmevAmixKm8sWFLOUzUNo07oKfHIuNwe+Csz/7p5jhjOOqZt5qoerjbTHwiw+JgYLiyNKJR0u5uBdzDgx5jMzcWlFXPVbnUhRCbC+PozqRb3bhbmIPL1YcmPZeibF6b23i9jik7KJunPfsEVI7gbKDGBx+Kg5VqJldEiM2LM5s37mCAyJXXIGtVKr8cegZy2ZxZdoUTSJWhItkEQ6O4wcU9LUe9jKKdSc4k2ZfJi6UmG7e0+LYy4rwlxSbpXV6kHW+KaacVrMet+65cbDc8RNxabcigLwU3WT6RAhrSUapj2jmKsFGBBSzTZuyJWfv1ZZaxRLQk/admNU1fvxZP3Q/pXtLc0/GWQG/CRdhd0K203IJ7/CJ4FCVaJemR10KaB7wexvaUAwyebn6hDtNVMf6g10m+7JrOc9WgKRULy3e982m0mh5v65bRcypvaYFdv7u9ckYmp60FxeIrQE7sfb1DKtysoYfcbsm2RLgUlKHhcMGJK2EzN0yOLw8H3pbYn3RzccAc6HYTKSQlT77mCTziVpQliGcHYQ0kk0GBr/QtOPjVn+fRvSXKG2OLibmHisV8SE9tinPy5sShwffnHS74SyJu9KKfhPwDlKkCmuXsFI2FR/J13Qim3TfiTMcTP4oEuzAKmZ+DQWJ+jgG/JSfcXfT09OQwjz5H8dJEp1Joc0ilDGXWRANEz/8C3yu3Og=="
  },
  "data/sst2_validation.tsv": {
    "sha256": "a7a3bfee7fc80566578791dfea290715934cc82f436c893361c4d0108a3554f8",
    "bytes": 99186,
    "zlib_base64": "..."
  },
  "data/sst2_validation.metadata.json": {
    "sha256": "d9c018c30511f87199684de5b848a8037e968412ec101db82677663228a4e1ef",
    "bytes": 566,
    "zlib_base64": "eNptUstugzAQvEfKP0RcW8L6gTF8R+/IGBssOTbCkLaK8u811JWI1Jtndmdn1trH+XS5ZL1YRFBL1lwy973mN2uLwa4qe9+r0jtthq0YwoITGSZrdsFdWBP1xrtUcZHlFf4FVnTKttKvbgmRf2xkpFF8U0rfE4QNYr6h50F3VMAeTg3R6Z6CpTnZ5IPZ2YNexVjrnqr9d1SOjiPQQTnqVhs3qHmajds3BIU7ACCko5zVRP/9wChwybYGUQnSaaUqLTmUjJUVr2rUayVwDRUqa0Kl5FhTwiSvCWFI0h4QcEHKkmqeBvb+01kv+la5fvLJfFyWKTRFMer8ZubZz1fpb0mgviY/L6pvxd6KAbMcQY7wByINkIbyK2YVovAG0AAkldfaSCNsG/w6S/Visg5D3F0LqaJNkc4iFC9HcT49fwDBxJUo"
  }
}


EXPECTED_ASSETS = {"sst2_instruction_eval.py", "data/sst2_validation.tsv",
                   "data/sst2_validation.metadata.json"}
assert set(EMBEDDED_ASSETS) == EXPECTED_ASSETS
for relative, item in EMBEDDED_ASSETS.items():
    raw = zlib.decompress(base64.b64decode(item["zlib_base64"]))
    assert hashlib.sha256(raw).hexdigest() == item["sha256"]
    assert len(raw) == item["bytes"]
    destination = ROOT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        if hashlib.sha256(destination.read_bytes()).hexdigest() != item["sha256"]:
            raise RuntimeError(f"已有文件 {destination} 与内置版本不同；请先保留并检查这个文件。")
    else:
        destination.write_bytes(raw)

with DATA.open(encoding="utf-8-sig", newline="") as handle:
    data_rows = list(csv.DictReader(handle, delimiter="\t"))
assert len(data_rows) == 872
assert sum(row["label"] == "1" for row in data_rows) == 444
assert sum(row["label"] == "0" for row in data_rows) == 428
dataset_meta = json.loads((DATA_DIR / "sst2_validation.metadata.json").read_text(encoding="utf-8"))
assert hashlib.sha256(DATA.read_bytes()).hexdigest() == dataset_meta["sha256"]
asset_manifest = {name: {key: value for key, value in item.items() if key != "zlib_base64"}
                  for name, item in EMBEDDED_ASSETS.items()}
(META_DIR / "embedded_assets.json").write_text(
    json.dumps(asset_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("验证集校验通过：872 条，正面 444 条，负面 428 条。")
print("数据 SHA-256：", dataset_meta["sha256"])


## 3. 检查 GPU

这里调用 `nvidia-smi`，并保存 GPU 信息。没有可用 NVIDIA GPU 时立即停止，
避免后面不知情地使用 CPU 运行全量实验。不要在本 Notebook 里安装或替换显卡驱动。


In [ ]:
if not shutil.which("nvidia-smi"):
    raise RuntimeError("未发现 NVIDIA GPU。请在 Kaggle Settings 选择可用 GPU 后重新启动会话。")
gpu_check = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_check.returncode != 0:
    raise RuntimeError("nvidia-smi 失败；请检查 Kaggle GPU 设置或额度：\n" + gpu_check.stderr)
print(gpu_check.stdout)
(META_DIR / "gpu_initial.txt").write_text(gpu_check.stdout, encoding="utf-8")


## 4. 安装固定版本 Ollama

从 [Ollama 官方地址](https://ollama.com/install.sh) 下载安装脚本，再用
`OLLAMA_VERSION=0.40.3` 指定版本；缺少 `curl` 或 `zstd` 时先通过 apt-get 补齐。
官方安装脚本可能显示 systemd 未运行的提示，Kaggle 后面使用独立进程启动服务。
安装方式与版本参数见 [官方 Linux 文档](https://docs.ollama.com/linux)，
固定版本见 [v0.40.3 发布页](https://github.com/ollama/ollama/releases/tag/v0.40.3)。


In [ ]:
missing_tools = [name for name in ("curl", "zstd") if not shutil.which(name)]
privilege_prefix = []
if os.geteuid() != 0:
    if not shutil.which("sudo"):
        raise RuntimeError("安装 Ollama 需要 root 或 sudo 权限。")
    privilege_prefix = ["sudo"]
if missing_tools:
    if not shutil.which("apt-get"):
        raise RuntimeError("缺少工具且找不到 apt-get：" + ", ".join(missing_tools))
    apt_env = dict(RUN_ENV, DEBIAN_FRONTEND="noninteractive")
    apt_update = subprocess.run(privilege_prefix + ["apt-get", "update", "-qq"], env=apt_env)
    if apt_update.returncode != 0:
        # 第三方软件源签名过期等问题会让 update 报错，但 Ubuntu 主源通常已更新，继续尝试安装。
        print("apt-get update 有报错，继续尝试安装：", ", ".join(missing_tools))
    subprocess.run(privilege_prefix + ["apt-get", "install", "-y", "-qq"] + missing_tools,
                   env=apt_env, check=True)
    still_missing = [name for name in missing_tools if not shutil.which(name)]
    if still_missing:
        raise RuntimeError("安装后仍缺少：" + ", ".join(still_missing))

installed_version = ""
if shutil.which("ollama"):
    existing = subprocess.run(["ollama", "--version"], capture_output=True, text=True, env=RUN_ENV)
    installed_version = existing.stdout + existing.stderr
if not re.search(r"(?<![0-9.])" + re.escape(OLLAMA_VERSION) + r"(?![0-9.])", installed_version):
    installer = Path("/tmp/teacher_sst2_ollama_install.sh")
    subprocess.run(["curl", "-fSL", "--retry", "3", "https://ollama.com/install.sh",
                    "-o", str(installer)], check=True, env=RUN_ENV)
    installer_meta = {"url": "https://ollama.com/install.sh", "requested_version": OLLAMA_VERSION,
                      "sha256": hashlib.sha256(installer.read_bytes()).hexdigest(),
                      "downloaded_at_utc": datetime.now(timezone.utc).isoformat()}
    (META_DIR / "ollama_install.json").write_text(
        json.dumps(installer_meta, indent=2), encoding="utf-8")
    install_env = dict(RUN_ENV, OLLAMA_VERSION=OLLAMA_VERSION)
    subprocess.run(privilege_prefix + ["env", "OLLAMA_VERSION=" + OLLAMA_VERSION,
                    "sh", str(installer)], env=install_env, check=True)
version_check = subprocess.run(["ollama", "--version"], capture_output=True, text=True, env=RUN_ENV)
CLI_VERSION = (version_check.stdout + version_check.stderr).strip()
print(CLI_VERSION)
if not re.search(r"(?<![0-9.])" + re.escape(OLLAMA_VERSION) + r"(?![0-9.])", CLI_VERSION):
    raise RuntimeError("Ollama 客户端版本不符合固定版本要求，请检查安装输出。")


## 5. 手动启动本地服务并等待就绪

服务只监听 `127.0.0.1:11434`，使用 `/tmp/teacher_sst2_models` 保存模型。
用独立的 `ollama serve` 进程启动，不依赖 systemd。
同一会话重跑此单元格会复用已经就绪且版本正确的服务。


In [ ]:
LOCAL_OPENER = urllib.request.build_opener(urllib.request.ProxyHandler({}))
def ollama_json(endpoint, payload=None, timeout=10):
    body = None if payload is None else json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(HOST + endpoint, data=body,
                                    headers={"Content-Type": "application/json"})
    with LOCAL_OPENER.open(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))

try:
    SERVER_VERSION = ollama_json("/api/version", timeout=2)
    print("复用本会话已就绪的 Ollama 服务。")
except (urllib.error.URLError, TimeoutError, OSError):
    _ollama_log = (META_DIR / "ollama_server.log").open("a", encoding="utf-8")
    _ollama_process = subprocess.Popen(["ollama", "serve"], env=RUN_ENV,
        stdout=_ollama_log, stderr=subprocess.STDOUT, start_new_session=True)
    deadline = time.monotonic() + 90
    while True:
        if _ollama_process.poll() is not None:
            tail = (META_DIR / "ollama_server.log").read_text(encoding="utf-8", errors="replace")[-4000:]
            raise RuntimeError("Ollama 服务启动失败：\n" + tail)
        try:
            SERVER_VERSION = ollama_json("/api/version", timeout=2)
            break
        except (urllib.error.URLError, TimeoutError, OSError):
            if time.monotonic() >= deadline:
                raise RuntimeError("等待 Ollama 服务超时，请查看 metadata/ollama_server.log。")
            time.sleep(1)
if SERVER_VERSION.get("version") != OLLAMA_VERSION:
    raise RuntimeError(f"服务版本 {SERVER_VERSION} 与固定版本 {OLLAMA_VERSION} 不一致。")
server_meta = {"endpoint": HOST, "server_version": SERVER_VERSION, "cli_version": CLI_VERSION,
               "models_directory": str(MODEL_DIR), "python": sys.version,
               "ollama_runtime_options": {key: RUN_ENV[key] for key in
                   ("OLLAMA_HOST", "OLLAMA_MODELS", "OLLAMA_NUM_PARALLEL", "OLLAMA_CONTEXT_LENGTH", "OLLAMA_KEEP_ALIVE")},
               "platform": platform.platform(), "checked_at_utc": datetime.now(timezone.utc).isoformat()}
(META_DIR / "server.json").write_text(json.dumps(server_meta, ensure_ascii=False, indent=2), encoding="utf-8")
print("服务已就绪：", HOST, "；版本：", SERVER_VERSION["version"])


## 6. 下载模型并记录身份

下载模型可能需要等待；重跑时会复用已下载的相同文件。
记录模型完整 digest、量化信息与 capabilities，方便汇报时说明实际使用的模型。
结果目录含模型 digest，模型标签更新时会分开保存实验，避免混用旧预测。
官方模型标签：[qwen3.5:4b](https://ollama.com/library/qwen3.5:4b)。


In [ ]:
subprocess.run(["ollama", "pull", MODEL], env=RUN_ENV, check=True)
tags = ollama_json("/api/tags")
model_entry = next((item for item in tags.get("models", [])
                    if item.get("name") == MODEL or item.get("model") == MODEL), None)
if model_entry is None or not model_entry.get("digest"):
    raise RuntimeError("下载后未找到模型完整 digest，请检查 ollama pull 输出。")
show = ollama_json("/api/show", {"model": MODEL})
capabilities = show.get("capabilities", [])
if capabilities and "completion" not in capabilities:
    raise RuntimeError("此模型不支持文本生成。")
if capabilities and THINK in ("on", "off") and "thinking" not in capabilities:
    raise RuntimeError("模型未声明 thinking 能力，不能把请求的思考开关当作有效实验条件。")
digest = model_entry["digest"]
digest_slug = re.sub(r"[^A-Za-z0-9]", "", digest.removeprefix("sha256:"))[:12]
model_slug = re.sub(r"[^A-Za-z0-9._-]+", "-", MODEL)
RESULTS_DIR = RESULT_ROOT / (model_slug + "_" + digest_slug)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
model_meta = {"model": MODEL, "digest": digest, "tag_entry": model_entry, "show": show,
              "ollama_version": SERVER_VERSION, "dataset_sha256": dataset_meta["sha256"],
              "prompt": PROMPT, "think_requested": THINK, "temperature": TEMPERATURE,
              "max_tokens": MAX_TOKENS, "workers": WORKERS,
              "results_directory": str(RESULTS_DIR),
              "recorded_at_utc": datetime.now(timezone.utc).isoformat()}
(META_DIR / ("model_" + model_slug + "_" + digest_slug + ".json")).write_text(
    json.dumps(model_meta, ensure_ascii=False, indent=2), encoding="utf-8")
print("模型：", MODEL, "；digest：", digest)
print("能力：", capabilities, "；详细信息：", show.get("details", {}))
print("结果目录：", RESULTS_DIR)


## 7. 运行例句与前 50 条

先用材料里的例句看模型输出，再跑前 50 条。
`50 条`只用于确认流程和速度，正式全量准确率应使用同样的 872 条验证集。
运行记录和错误输出均保存在结果目录；同一设置重跑会接续 JSONL 断点。
停止正在运行的单元格时，下面的助手会尝试向评测子进程发送 Ctrl+C，保留已写入的断点。


In [ ]:
BASE_COMMAND = [sys.executable, str(SCRIPT), "--provider", "ollama", "--base-url", HOST,
                "--model", MODEL, "--prompt", PROMPT, "--think", THINK,
                "--data-file", str(DATA), "--temperature", str(TEMPERATURE),
                "--max-tokens", str(MAX_TOKENS), "--workers", str(WORKERS),
                "--timeout", str(REQUEST_TIMEOUT), "--out-dir", str(RESULTS_DIR)]

def run_evaluator(extra_args, label):
    command = BASE_COMMAND + list(extra_args)
    started = time.monotonic()
    log_path = RESULTS_DIR / (label + "_console.log")
    with log_path.open("a", encoding="utf-8") as log:
        log.write("\n" + datetime.now(timezone.utc).isoformat() + "\n")
        process = subprocess.Popen(command, cwd=ROOT, env=RUN_ENV, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace",
            bufsize=1, start_new_session=True)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            return_code = process.wait()
        except KeyboardInterrupt:
            if process.poll() is None:
                process.send_signal(signal.SIGINT)
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.terminate()
                    process.wait(timeout=10)
            print("评测已停止；同一设置重跑会接续已保存的断点。")
            raise
        finally:
            process.stdout.close()
    elapsed = time.monotonic() - started
    if return_code != 0:
        raise RuntimeError(f"评测退出码 {return_code}；请查看 {log_path}。")
    print(f"本次单元格请求耗时：{elapsed:.1f} 秒。")
    return elapsed

run_evaluator(["--demo"], "demo")
ps_after_demo = ollama_json("/api/ps")
(META_DIR / "ollama_ps_after_demo.json").write_text(
    json.dumps(ps_after_demo, ensure_ascii=False, indent=2), encoding="utf-8")
loaded_model = next((item for item in ps_after_demo.get("models", [])
                     if item.get("name") == MODEL or item.get("model") == MODEL
                     or item.get("digest") == digest), None)
if loaded_model is None or int(loaded_model.get("size_vram", 0) or 0) == 0:
    server_log = META_DIR / "ollama_server.log"
    tail = server_log.read_text(encoding="utf-8", errors="replace")[-4000:] if server_log.exists() else "没有本 Notebook 的服务日志。"
    raise RuntimeError("Ollama 未报告当前模型驻留 GPU，已停止，避免用 CPU 继续跑 50 条。"
                       "请检查 Kaggle GPU 设置和下面的服务日志；不要只依据 nvidia-smi 判断推理使用了 GPU。\n" + tail)
print(f"Ollama 已报告模型使用 GPU 显存：{loaded_model['size_vram'] / 1024**3:.2f} GiB。")


In [ ]:
SAMPLE50_SECONDS = run_evaluator(["--n", "50"], "n50")
ps_after50 = ollama_json("/api/ps")
(META_DIR / "ollama_ps_after50.json").write_text(
    json.dumps(ps_after50, ensure_ascii=False, indent=2), encoding="utf-8")
ps_result = subprocess.run(["ollama", "ps"], capture_output=True, text=True, env=RUN_ENV, check=True)
print(ps_result.stdout)
(META_DIR / "ollama_ps_after50.txt").write_text(ps_result.stdout, encoding="utf-8")
after_gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True)
(META_DIR / "gpu_after50.txt").write_text(after_gpu.stdout, encoding="utf-8")

candidates = []
for path in RESULTS_DIR.glob("*_summary.json"):
    summary = json.loads(path.read_text(encoding="utf-8"))
    if (summary.get("n") == 50 and summary.get("model") == MODEL
            and summary.get("prompt") == PROMPT and summary.get("think") == THINK
            and summary.get("temperature") == TEMPERATURE and summary.get("max_tokens") == MAX_TOKENS):
        candidates.append((path.stat().st_mtime_ns, summary))
if not candidates:
    raise RuntimeError("没有找到本设置的 50 条完整汇总。")
sample_summary = max(candidates, key=lambda pair: pair[0])[1]
print(f"50 条准确率：{sample_summary['correct']}/50 = {sample_summary['accuracy']:.2%}")
print("无效输出：", sample_summary["invalid"], "；请求失败：", sample_summary["errors"],
      "；截断：", sample_summary["truncated"])
print(f"按单条平均请求时间粗估 872 条约需 {sample_summary['avg_seconds'] * 872 / 60:.1f} 分钟。")
print("这是速度粗估；请结合 ollama ps 的实际 GPU/CPU 状态决定是否继续。")


## 8. 可选：全量 872 条

检查上面的速度、模型输出、请求失败与截断情况后，设置 `RUN_FULL = True` 再执行本单元格。
默认关闭；同设置重跑不加 `--fresh`，保留已经完成的请求。
若需要更小模型，请修改 `MODEL` 并重新执行模型下载、例句和 50 条单元格。


In [ ]:
if RUN_FULL:
    run_evaluator(["--n", "0"], "full872")
else:
    print("RUN_FULL=False，已跳过全量。检查 50 条速度后，将 RUN_FULL 改为 True 再运行本单元格。")


## 9. 汇总、导出错例与下载结果包

汇总表会明确列出样本数。只有 872 条完整结果适合与 API 和 DeBERTa 的 872 条验证集结果比较。
每个 CSV 生成对应的 `_errors.csv`，只包含 `correct=0` 的错例。
ZIP 按白名单收集 results、data、metadata 和评测脚本；不包含模型权重或凭据文件。
在 Kaggle Output 面板下载 `teacher_task_ollama_results.zip`，之后放回本机 teacher_task 供整理。
新会话的 `/kaggle/working` 未必保留上次文件，结束前请下载输出包。


In [ ]:
run_evaluator(["--report"], "report")
for csv_path in sorted(RESULTS_DIR.glob("*.csv")):
    if csv_path.stem.endswith("_errors"):
        continue
    with csv_path.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        fieldnames = reader.fieldnames
        errors = [row for row in reader if row.get("correct") == "0"]
    error_path = csv_path.with_name(csv_path.stem + "_errors.csv")
    with error_path.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(errors)
    print(error_path.name, "：", len(errors), "条错例")

allowed_files = []
for directory in (RESULT_ROOT, DATA_DIR, META_DIR):
    for path in directory.rglob("*"):
        if path.is_file() and not path.is_symlink():
            if path == META_DIR / "export_manifest.json":
                continue  # 清单不记录自己的哈希；重跑时也不纳入上次的旧清单。
            if path.suffix.lower() not in {".json", ".jsonl", ".csv", ".tsv", ".md", ".txt", ".log"}:
                continue
            if any(part.startswith(".") for part in path.relative_to(ROOT).parts):
                continue
            allowed_files.append(path)
allowed_files.append(SCRIPT)
export_manifest = [{"path": str(path.relative_to(ROOT)), "bytes": path.stat().st_size,
                    "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
                   for path in sorted(set(allowed_files))]
manifest_path = META_DIR / "export_manifest.json"
manifest_path.write_text(json.dumps(export_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
allowed_files.append(manifest_path)
zip_path = Path("/kaggle/working/teacher_task_ollama_results.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(set(allowed_files)):
        archive.write(path, arcname="teacher_task/" + path.relative_to(ROOT).as_posix())
print("结果包：", zip_path, "；大小：", round(zip_path.stat().st_size / 1024, 1), "KB")
print("结果目录：", RESULTS_DIR)
